In [1]:
import os
import glob
import pandas as pd

def inspect_biomass_folder(file_path):
    """
    Dynamically locates the table header in raw or preprocessed files, loads the CSV,
    and outputs shape, column structure, and baseline statistics.
    """
    header_row_index = 0
    
    # Locate the telemetry table header row (works for both raw Agilent and _cleaned files)
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            # We look for standard raw headers or preprocessed headers like 'timestamp' / 'scan_num'
            if any(key in line.lower() for key in ['scan num', '101 (', 'scan swee', 'timestamp', 'scan_num']):
                header_row_index = idx
                break
                
    # Load dataset from the detected header row
    df = pd.read_csv(file_path, skiprows=header_row_index)
    
    # Clean column whitespace and drop completely empty rows or columns
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    return df

# ==========================================
# EXECUTION ON BIOMASS FOLDER
# ==========================================
biomass_path = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\Biomass"

# We strictly match only .csv files, automatically ignoring '.keep' and other non-data files
csv_files = glob.glob(os.path.join(biomass_path, "*.csv"))

print(f"Found {len(csv_files)} valid CSV files in Biomass folder (ignoring .keep). Running inspection...\n")

for i, file_path in enumerate(csv_files, 1):
    file_name = os.path.basename(file_path)
    
    try:
        df = inspect_biomass_folder(file_path)
        
        print(f"File {i}: {file_name}")
        print(f"   Shape: {df.shape[0]} rows by {df.shape[1]} columns")
        print(f"   Columns: {list(df.columns)}")
        print(f"   Missing Values: {df.isnull().sum().sum()} total nulls")
        
        # Select numeric columns for basic baseline stats
        numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
        if len(numeric_cols) > 0:
            print("\n   Baseline Data Summary (First 5 Numeric Columns):")
            print(df[numeric_cols[:5]].describe().loc[['mean', 'min', 'max', 'std']])
        
        print("\n" + "="*65 + "\n")
        
    except Exception as e:
        print(f"Could not read {file_name}: {e}\n")

Found 6 valid CSV files in Biomass folder (ignoring .keep). Running inspection...

File 1: 1781367912_gasifier 50slpm 0.csv
   Shape: 796 rows by 7 columns
   Columns: ['Scan Sweep Time (Sec)', 'Scan Number', '101 (°C)', '102 (°C)', '103 (°C)', '104 (°C)', '105 (°C)']
   Missing Values: 0 total nulls

   Baseline Data Summary (First 5 Numeric Columns):
      Scan Number     101 (°C)     102 (°C)    103 (°C)    104 (°C)
mean   398.500000   425.204070   495.063349  522.364208  520.175080
min      1.000000    27.770788    27.511129   30.257773   30.520025
max    796.000000  1171.626450  1107.979790  989.705587  889.174997
std    229.929699   427.698895   432.131571  359.681821  254.000610


File 2: 20260613T100035_Gasifier 1 0_cleaned.csv
   Shape: 714 rows by 9 columns
   Columns: ['timestamp', 'scan_number', 'ch_00', 'ch_01', 'ch_02', 'ch_03', 'ch_04', 'system_file', 'system_id']
   Missing Values: 0 total nulls

   Baseline Data Summary (First 5 Numeric Columns):
      scan_number     